# 2.数据清洗

## 2.1 工作目录整理

In [1]:
# ========== 简化版：只创建文件夹 ==========
import os

# 定义需要创建的文件夹
folders = [
    "data/stock",
    "data/index", 
    "data/macro",
    "data/finance",
    "data/clean",
    "data/combined",
    "output",
]

print("创建文件夹...")
for folder in folders:
    if not os.path.exists(folder):
        os.makedirs(folder)
        print(f"  ✓ 创建: {folder}")
    else:
        print(f"  ○ 已存在: {folder}")

print("\n✅ 完成！")

创建文件夹...
  ○ 已存在: data/stock
  ○ 已存在: data/index
  ○ 已存在: data/macro
  ○ 已存在: data/finance
  ○ 已存在: data/clean
  ○ 已存在: data/combined
  ○ 已存在: output

✅ 完成！


In [2]:
# ========== 删除文件名中的中文 ==========
print("="*60)
print("遍历文件夹，删除文件名中的中文")
print("="*60)

import os
import re

# 定义需要处理的根目录
root_dir = "."  # 当前目录，可修改为具体路径如 "data"

# 定义正则表达式：匹配中文字符（包括繁简体）
chinese_pattern = re.compile(r'[\u4e00-\u9fff\u3400-\u4dbf\uf900-\ufaff\u3000-\u303f\uff00-\uffef]+')

# 统计信息
total_renamed = 0
total_skipped = 0

def remove_chinese_from_filename(filename):
    """删除文件名中的中文字符"""
    # 分离文件名和扩展名
    name, ext = os.path.splitext(filename)
    # 删除中文
    new_name = chinese_pattern.sub('', name)
    # 如果删除中文后为空，则使用原名称
    if not new_name:
        new_name = name
    # 去除多余的下划线（连续下划线变为单个）
    new_name = re.sub(r'_+', '_', new_name)
    # 去掉开头和结尾的下划线
    new_name = new_name.strip('_')
    return new_name + ext

def rename_files_in_directory(directory):
    """遍历目录并重命名文件"""
    global total_renamed, total_skipped
    
    for root, dirs, files in os.walk(directory):
        for filename in files:
            # 检查文件名是否包含中文
            if chinese_pattern.search(filename):
                old_path = os.path.join(root, filename)
                new_filename = remove_chinese_from_filename(filename)
                new_path = os.path.join(root, new_filename)
                
                # 如果新文件名与原文件名相同，跳过
                if new_filename == filename:
                    total_skipped += 1
                    continue
                
                # 如果新文件名已存在，添加数字后缀
                counter = 1
                original_new_path = new_path
                while os.path.exists(new_path):
                    name, ext = os.path.splitext(original_new_path)
                    new_path = f"{name}_{counter}{ext}"
                    counter += 1
                
                try:
                    os.rename(old_path, new_path)
                    print(f"  ✓ 重命名: {filename}")
                    print(f"       → {new_filename}")
                    total_renamed += 1
                except Exception as e:
                    print(f"  ✗ 失败: {filename} - {e}")
            else:
                total_skipped += 1

# 执行重命名
print(f"\n📁 扫描目录: {os.path.abspath(root_dir)}")
print("-"*60)

rename_files_in_directory(root_dir)

print("-"*60)
print(f"\n✅ 完成！")
print(f"  重命名文件数: {total_renamed}")
print(f"  无需处理文件数: {total_skipped}")

遍历文件夹，删除文件名中的中文

📁 扫描目录: d:\Desktop\第二次作业
------------------------------------------------------------
------------------------------------------------------------

✅ 完成！
  重命名文件数: 0
  无需处理文件数: 41


In [3]:
# ========== 批量修正所有 index 文件名 ==========
import os
import re

print("="*60)
print("批量修正指数文件名")
print("="*60)

# 搜索的根目录
search_dirs = [".", "data", "data/index"]

# 匹配模式：index_代码_名称.csv → index_代码.csv
pattern = re.compile(r'index_(\d{6})_.+\.csv')

renamed_count = 0

for search_dir in search_dirs:
    if not os.path.exists(search_dir):
        continue
    
    for root, dirs, files in os.walk(search_dir):
        for filename in files:
            match = pattern.match(filename)
            if match:
                code = match.group(1)
                new_filename = f"index_{code}.csv"
                
                if new_filename != filename:
                    old_path = os.path.join(root, filename)
                    new_path = os.path.join(root, new_filename)
                    
                    # 如果目标文件已存在，添加后缀
                    counter = 1
                    original_new_path = new_path
                    while os.path.exists(new_path):
                        name, ext = os.path.splitext(original_new_path)
                        new_path = f"{name}_{counter}{ext}"
                        counter += 1
                    
                    try:
                        os.rename(old_path, new_path)
                        print(f"✓ {filename} → {os.path.basename(new_path)}")
                        renamed_count += 1
                    except Exception as e:
                        print(f"✗ 失败: {filename} - {e}")

if renamed_count == 0:
    print("\n未找到需要重命名的 index 文件。")
    print("当前 index 文件列表：")
    for search_dir in search_dirs:
        if os.path.exists(search_dir):
            for root, dirs, files in os.walk(search_dir):
                for f in files:
                    if f.startswith('index_'):
                        print(f"  - {f}")
else:
    print(f"\n✅ 完成！共重命名 {renamed_count} 个文件")

批量修正指数文件名

未找到需要重命名的 index 文件。
当前 index 文件列表：
  - index_000300.csv
  - index_000905.csv
  - index_000300.csv
  - index_000905.csv
  - index_000300.csv
  - index_000905.csv


In [4]:
# ========== 简化版：快速查看主要结构 ==========
import os

print("="*60)
print("项目主要文件结构")
print("="*60)

def list_structure(directory, indent=0, max_depth=2):
    if indent > max_depth:
        return
    
    try:
        items = sorted(os.listdir(directory))
    except:
        return
    
    # 过滤隐藏文件和缓存
    items = [i for i in items if not i.startswith('.')]
    items = [i for i in items if i not in ['__pycache__', '.ipynb_checkpoints']]
    
    for i, item in enumerate(items):
        item_path = os.path.join(directory, item)
        is_last = (i == len(items) - 1)
        
        prefix = "    " * indent
        branch = "└── " if is_last else "├── "
        
        if os.path.isdir(item_path):
            print(f"{prefix}{branch}📁 {item}/")
            if indent < max_depth:
                list_structure(item_path, indent + 1, max_depth)
        else:
            print(f"{prefix}{branch}📄 {item}")

print("📁 ./")
list_structure(".")

print("\n" + "="*60)

项目主要文件结构
📁 ./
├── 📄 01_download.ipynb
├── 📄 02_clean.ipynb
├── 📄 03_analysis.ipynb
├── 📄 README.md
├── 📁 data/
    ├── 📁 clean/
        ├── 📄 stock_clean.csv
        └── 📄 stock_clean.parquet
    ├── 📁 combined/
        └── 📄 combined_data.csv
    ├── 📁 finance/
        └── 📄 finance_ratios.csv
    ├── 📁 index/
        ├── 📄 index_000300.csv
        └── 📄 index_000905.csv
    ├── 📁 macro/
        ├── 📄 macro_cpi.csv
        ├── 📄 macro_lpr.csv
        └── 📄 macro_m2.csv
    └── 📁 stock/
        ├── 📄 stock_000002.csv
        ├── 📄 stock_000002_clean.csv
        ├── 📄 stock_000625.csv
        ├── 📄 stock_000625_clean.csv
        ├── 📄 stock_000858.csv
        ├── 📄 stock_000858_clean.csv
        ├── 📄 stock_002352.csv
        ├── 📄 stock_002352_clean.csv
        ├── 📄 stock_600036.csv
        ├── 📄 stock_600036_clean.csv
        ├── 📄 stock_600050.csv
        ├── 📄 stock_600050_clean.csv
        ├── 📄 stock_600519.csv
        ├── 📄 stock_600519_clean.csv
        ├── 📄 stock_601166.csv
 

## 2.2 数据清洗与处理

In [1]:
# ========== 导入库 ==========
import pandas as pd
import numpy as np
import os
import glob
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("库导入成功！")
print(f"当前工作目录: {os.getcwd()}")

库导入成功！
当前工作目录: d:\Desktop\第二次作业


In [2]:
# ========== 加载所有股票数据 ==========
print("="*80)
print("3.1 单表清洗 - 加载股票数据")
print("="*80)

# 定义路径
stock_folder = "data/stock"
if not os.path.exists(stock_folder):
    print(f"❌ 文件夹不存在: {stock_folder}")
    print("请先运行 01_download.ipynb 下载数据")
else:
    # 获取所有股票CSV文件
    stock_files = glob.glob(os.path.join(stock_folder, "stock_*.csv"))
    print(f"找到 {len(stock_files)} 个股票数据文件")

# 存储清洗后的数据
cleaned_stocks = {}
cleaning_log = []

for file_path in stock_files:
    filename = os.path.basename(file_path)
    code = filename.replace("stock_", "").replace(".csv", "")
    
    print(f"\n处理: {filename} (代码: {code})")
    
    # 读取数据
    df = pd.read_csv(file_path, encoding='utf-8-sig')
    print(f"  原始数据形状: {df.shape}")

3.1 单表清洗 - 加载股票数据
找到 10 个股票数据文件

处理: stock_000002.csv (代码: 000002)
  原始数据形状: (1517, 7)

处理: stock_000625.csv (代码: 000625)
  原始数据形状: (1517, 7)

处理: stock_000858.csv (代码: 000858)
  原始数据形状: (1517, 7)

处理: stock_002352.csv (代码: 002352)
  原始数据形状: (1517, 7)

处理: stock_600036.csv (代码: 600036)
  原始数据形状: (1517, 7)

处理: stock_600050.csv (代码: 600050)
  原始数据形状: (1517, 7)

处理: stock_600519.csv (代码: 600519)
  原始数据形状: (1517, 7)

处理: stock_601166.csv (代码: 601166)
  原始数据形状: (1517, 7)

处理: stock_601633.csv (代码: 601633)
  原始数据形状: (1517, 7)

处理: stock_601857.csv (代码: 601857)
  原始数据形状: (1517, 7)


In [7]:
# ========== 缺失值检测 ==========
print("\n" + "-"*40)
print("1. 缺失值检测")
print("-"*40)

missing_stats = []

for file_path in stock_files:
    filename = os.path.basename(file_path)
    code = filename.replace("stock_", "").replace(".csv", "")
    
    df = pd.read_csv(file_path, encoding='utf-8-sig')
    
    # 统计缺失值
    missing_count = df.isnull().sum()
    missing_pct = (df.isnull().sum() / len(df)) * 100
    
    missing_df = pd.DataFrame({
        '列名': missing_count.index,
        '缺失数量': missing_count.values,
        '缺失比例(%)': missing_pct.values
    })
    
    missing_df = missing_df[missing_df['缺失数量'] > 0]
    
    if len(missing_df) > 0:
        missing_stats.append({
            '代码': code,
            '文件名': filename,
            '缺失统计': missing_df.to_dict('records')
        })
        print(f"\n  {code}: 发现缺失值")
        print(missing_df.to_string(index=False))
    else:
        print(f"  {code}: 无缺失值 ✓")

# 汇总说明
print("\n" + "="*60)
print("缺失值分析说明：")
print("="*60)
print("""
【缺失值可能原因】
1. 停牌：股票因重大事项暂停交易，期间无行情数据
2. 节假日：A股市场休市（春节、国庆等），无交易日
3. 数据源问题：baostock 接口返回时可能个别字段为空
4. 新股上市：上市日期晚于2020-01-01，前期无数据

【处理策略说明】
- 成交量、成交额：向前填充（ffill），因为停牌期间无交易，
  用最近交易日的值填充不影响分析（实际停牌期间不应交易）
- 价格数据（开/收/高/低）：向前填充，理由同上
- 如果缺失比例 > 30%，考虑删除该行（但本次数据无此情况）
""")


----------------------------------------
1. 缺失值检测
----------------------------------------
  000002: 无缺失值 ✓
  000002_clean: 无缺失值 ✓
  000625: 无缺失值 ✓
  000625_clean: 无缺失值 ✓
  000858: 无缺失值 ✓
  000858_clean: 无缺失值 ✓
  002352: 无缺失值 ✓
  002352_clean: 无缺失值 ✓
  600036: 无缺失值 ✓
  600036_clean: 无缺失值 ✓
  600050: 无缺失值 ✓
  600050_clean: 无缺失值 ✓
  600519: 无缺失值 ✓
  600519_clean: 无缺失值 ✓
  601166: 无缺失值 ✓
  601166_clean: 无缺失值 ✓
  601633: 无缺失值 ✓
  601633_clean: 无缺失值 ✓
  601857: 无缺失值 ✓
  601857_clean: 无缺失值 ✓

缺失值分析说明：

【缺失值可能原因】
1. 停牌：股票因重大事项暂停交易，期间无行情数据
2. 节假日：A股市场休市（春节、国庆等），无交易日
3. 数据源问题：baostock 接口返回时可能个别字段为空
4. 新股上市：上市日期晚于2020-01-01，前期无数据

【处理策略说明】
- 成交量、成交额：向前填充（ffill），因为停牌期间无交易，
  用最近交易日的值填充不影响分析（实际停牌期间不应交易）
- 价格数据（开/收/高/低）：向前填充，理由同上
- 如果缺失比例 > 30%，考虑删除该行（但本次数据无此情况）



In [8]:
# ========== 缺失值处理 ==========
print("\n" + "-"*40)
print("2. 缺失值处理（向前填充）")
print("-"*40)

def fill_missing_values(df):
    """向前填充缺失值"""
    original_missing = df.isnull().sum().sum()
    
    # 向前填充（用前一个有效值填充）
    df = df.fillna(method='ffill')
    
    # 如果第一行有缺失，向后填充
    df = df.fillna(method='bfill')
    
    after_missing = df.isnull().sum().sum()
    filled_count = original_missing - after_missing
    
    return df, filled_count

for file_path in stock_files:
    filename = os.path.basename(file_path)
    code = filename.replace("stock_", "").replace(".csv", "")
    
    df = pd.read_csv(file_path, encoding='utf-8-sig')
    original_missing = df.isnull().sum().sum()
    
    if original_missing > 0:
        df, filled = fill_missing_values(df)
        print(f"  {code}: 填充了 {filled} 个缺失值")
        
        # 保存清洗后的数据（临时）
        df.to_csv(file_path.replace('.csv', '_temp.csv'), index=False, encoding='utf-8-sig')
    else:
        print(f"  {code}: 无需填充")

print("\n【选择依据】")
print("采用向前填充(ffill)而非删除的原因：")
print("1. 股票数据缺失通常是因为停牌，用最近交易日数据填充最合理")
print("2. 删除含缺失值的行会破坏时间序列的连续性")
print("3. 停牌期间股票无法交易，用前值表示'价格不变'符合实际")


----------------------------------------
2. 缺失值处理（向前填充）
----------------------------------------
  000002: 无需填充
  000002_clean: 无需填充
  000625: 无需填充
  000625_clean: 无需填充
  000858: 无需填充
  000858_clean: 无需填充
  002352: 无需填充
  002352_clean: 无需填充
  600036: 无需填充
  600036_clean: 无需填充
  600050: 无需填充
  600050_clean: 无需填充
  600519: 无需填充
  600519_clean: 无需填充
  601166: 无需填充
  601166_clean: 无需填充
  601633: 无需填充
  601633_clean: 无需填充
  601857: 无需填充
  601857_clean: 无需填充

【选择依据】
采用向前填充(ffill)而非删除的原因：
1. 股票数据缺失通常是因为停牌，用最近交易日数据填充最合理
2. 删除含缺失值的行会破坏时间序列的连续性
3. 停牌期间股票无法交易，用前值表示'价格不变'符合实际


In [9]:
# ========== 日期格式统一 ==========
print("\n" + "-"*40)
print("3. 日期格式统一")
print("-"*40)

for file_path in stock_files:
    filename = os.path.basename(file_path)
    code = filename.replace("stock_", "").replace(".csv", "")
    
    # 读取临时文件或原文件
    temp_path = file_path.replace('.csv', '_temp.csv')
    if os.path.exists(temp_path):
        df = pd.read_csv(temp_path, encoding='utf-8-sig')
    else:
        df = pd.read_csv(file_path, encoding='utf-8-sig')
    
    # 转换日期格式
    print(f"\n  {code} 清洗前日期类型: {df['日期'].dtype}")
    print(f"  日期示例: {df['日期'].iloc[:3].tolist()}")
    
    df['日期'] = pd.to_datetime(df['日期'])
    df.set_index('日期', inplace=True)
    
    print(f"  清洗后: 索引类型 {df.index.dtype}, 范围 {df.index.min()} 至 {df.index.max()}")
    
    # 保存
    df.to_csv(file_path.replace('.csv', '_clean.csv'), encoding='utf-8-sig')
    cleaned_stocks[code] = df

print("\n【变化说明】")
print("清洗前: 日期列为 object/string 类型，格式如 '2020-01-02'")
print("清洗后: 日期列转为 datetime64 类型，并设为 DataFrame 索引")
print("优势: 便于时间序列操作、切片、重采样等")


----------------------------------------
3. 日期格式统一
----------------------------------------

  000002 清洗前日期类型: object
  日期示例: ['2020-01-02', '2020-01-03', '2020-01-06']
  清洗后: 索引类型 datetime64[ns], 范围 2020-01-02 00:00:00 至 2026-04-08 00:00:00

  000002_clean 清洗前日期类型: object
  日期示例: ['2020-01-02', '2020-01-03', '2020-01-06']
  清洗后: 索引类型 datetime64[ns], 范围 2020-01-02 00:00:00 至 2026-04-08 00:00:00

  000625 清洗前日期类型: object
  日期示例: ['2020-01-02', '2020-01-03', '2020-01-06']
  清洗后: 索引类型 datetime64[ns], 范围 2020-01-02 00:00:00 至 2026-04-08 00:00:00

  000625_clean 清洗前日期类型: object
  日期示例: ['2020-01-02', '2020-01-03', '2020-01-06']
  清洗后: 索引类型 datetime64[ns], 范围 2020-01-02 00:00:00 至 2026-04-08 00:00:00

  000858 清洗前日期类型: object
  日期示例: ['2020-01-02', '2020-01-03', '2020-01-06']
  清洗后: 索引类型 datetime64[ns], 范围 2020-01-02 00:00:00 至 2026-04-08 00:00:00

  000858_clean 清洗前日期类型: object
  日期示例: ['2020-01-02', '2020-01-03', '2020-01-06']
  清洗后: 索引类型 datetime64[ns], 范围 2020-01-02 00:00:00 至 2026-04-0

In [10]:
# ========== 数据类型检查 ==========
print("\n" + "-"*40)
print("4. 数据类型检查与转换")
print("-"*40)

for code, df in cleaned_stocks.items():
    print(f"\n  {code}:")
    
    # 检查各列类型
    numeric_cols = ['开盘', '最高', '最低', '收盘', '成交量', '成交额']
    
    for col in numeric_cols:
        if col in df.columns:
            original_type = df[col].dtype
            if not pd.api.types.is_numeric_dtype(df[col]):
                print(f"    转换前 {col}: {original_type}")
                df[col] = pd.to_numeric(df[col], errors='coerce')
                print(f"    转换后 {col}: {df[col].dtype}")
            else:
                print(f"    ✓ {col}: {original_type} (已是数值型)")
    
    # 更新
    cleaned_stocks[code] = df

print("\n【说明】")
print("价格、成交量字段应为数值类型以便计算收益率等指标")
print("若发现字符型(如'--'、'None')，用 pd.to_numeric 强制转换，无法转换的变为 NaN")


----------------------------------------
4. 数据类型检查与转换
----------------------------------------

  000002:
    ✓ 开盘: float64 (已是数值型)
    ✓ 最高: float64 (已是数值型)
    ✓ 最低: float64 (已是数值型)
    ✓ 收盘: float64 (已是数值型)
    ✓ 成交量: int64 (已是数值型)
    ✓ 成交额: float64 (已是数值型)

  000002_clean:
    ✓ 开盘: float64 (已是数值型)
    ✓ 最高: float64 (已是数值型)
    ✓ 最低: float64 (已是数值型)
    ✓ 收盘: float64 (已是数值型)
    ✓ 成交量: int64 (已是数值型)
    ✓ 成交额: float64 (已是数值型)

  000625:
    ✓ 开盘: float64 (已是数值型)
    ✓ 最高: float64 (已是数值型)
    ✓ 最低: float64 (已是数值型)
    ✓ 收盘: float64 (已是数值型)
    ✓ 成交量: int64 (已是数值型)
    ✓ 成交额: float64 (已是数值型)

  000625_clean:
    ✓ 开盘: float64 (已是数值型)
    ✓ 最高: float64 (已是数值型)
    ✓ 最低: float64 (已是数值型)
    ✓ 收盘: float64 (已是数值型)
    ✓ 成交量: int64 (已是数值型)
    ✓ 成交额: float64 (已是数值型)

  000858:
    ✓ 开盘: float64 (已是数值型)
    ✓ 最高: float64 (已是数值型)
    ✓ 最低: float64 (已是数值型)
    ✓ 收盘: float64 (已是数值型)
    ✓ 成交量: int64 (已是数值型)
    ✓ 成交额: float64 (已是数值型)

  000858_clean:
    ✓ 开盘: float64 (已是数值型)
    ✓ 最高: floa

In [11]:
# ========== 重复值检测与处理 ==========
print("\n" + "-"*40)
print("5. 重复值检测与删除")
print("-"*40)

total_duplicates = 0

for code, df in cleaned_stocks.items():
    # 检查重复行（基于所有列）
    duplicate_count = df.duplicated().sum()
    
    if duplicate_count > 0:
        print(f"  {code}: 发现 {duplicate_count} 个重复行")
        df = df.drop_duplicates()
        total_duplicates += duplicate_count
        cleaned_stocks[code] = df
    else:
        print(f"  {code}: 无重复行 ✓")

print(f"\n总计删除重复行数: {total_duplicates}")

print("\n【说明】")
print("重复行可能原因：")
print("1. 数据源重复推送同一天的数据")
print("2. 多次下载导致重复记录")
print("处理方式：基于所有列删除重复行，保留第一次出现")


----------------------------------------
5. 重复值检测与删除
----------------------------------------
  000002: 无重复行 ✓
  000002_clean: 无重复行 ✓
  000625: 无重复行 ✓
  000625_clean: 无重复行 ✓
  000858: 无重复行 ✓
  000858_clean: 无重复行 ✓
  002352: 发现 2 个重复行
  002352_clean: 发现 2 个重复行
  600036: 无重复行 ✓
  600036_clean: 无重复行 ✓
  600050: 无重复行 ✓
  600050_clean: 无重复行 ✓
  600519: 无重复行 ✓
  600519_clean: 无重复行 ✓
  601166: 无重复行 ✓
  601166_clean: 无重复行 ✓
  601633: 无重复行 ✓
  601633_clean: 无重复行 ✓
  601857: 无重复行 ✓
  601857_clean: 无重复行 ✓

总计删除重复行数: 4

【说明】
重复行可能原因：
1. 数据源重复推送同一天的数据
2. 多次下载导致重复记录
处理方式：基于所有列删除重复行，保留第一次出现


In [12]:
# ========== 离群值标注 ==========
print("\n" + "-"*40)
print("6. 离群值标注（日收益率 ±20%）")
print("-"*40)

extreme_stats = []

for code, df in cleaned_stocks.items():
    # 计算日收益率
    df['return'] = df['收盘'].pct_change() * 100
    
    # 标注极端值
    df['is_extreme'] = (df['return'].abs() > 20)
    
    extreme_count = df['is_extreme'].sum()
    extreme_stats.append({
        '代码': code,
        '极端值数量': extreme_count,
        '极端值比例(%)': round(extreme_count / len(df) * 100, 4)
    })
    
    if extreme_count > 0:
        print(f"  {code}: 发现 {extreme_count} 个极端值交易日")
        # 显示极端值示例
        extreme_dates = df[df['is_extreme']].index[:3].strftime('%Y-%m-%d').tolist()
        print(f"    示例日期: {extreme_dates}")
    else:
        print(f"  {code}: 无极端值")
    
    cleaned_stocks[code] = df

# 统计表格
extreme_df = pd.DataFrame(extreme_stats)
print("\n极端值统计表：")
print(extreme_df.to_string(index=False))

print("\n【可能成因分析】")
print("单日涨跌幅超过20%的可能原因：")
print("1. 除权除息：股票分红、送股导致价格大幅调整")
print("2. 重大利好/利空：重组、业绩暴增、黑天鹅事件")
print("3. 新股上市：首日无涨跌幅限制")
print("4. 数据错误：复权计算异常")
print("\n【处理方式】")
print("标注为 is_extreme=True 但不删除，保留用于后续分析")
print("在回归分析时可选择剔除或保留观察")


----------------------------------------
6. 离群值标注（日收益率 ±20%）
----------------------------------------
  000002: 无极端值
  000002_clean: 无极端值
  000625: 无极端值
  000625_clean: 无极端值
  000858: 无极端值
  000858_clean: 无极端值
  002352: 无极端值
  002352_clean: 无极端值
  600036: 无极端值
  600036_clean: 无极端值
  600050: 无极端值
  600050_clean: 无极端值
  600519: 无极端值
  600519_clean: 无极端值
  601166: 无极端值
  601166_clean: 无极端值
  601633: 无极端值
  601633_clean: 无极端值
  601857: 无极端值
  601857_clean: 无极端值

极端值统计表：
          代码  极端值数量  极端值比例(%)
      000002      0       0.0
000002_clean      0       0.0
      000625      0       0.0
000625_clean      0       0.0
      000858      0       0.0
000858_clean      0       0.0
      002352      0       0.0
002352_clean      0       0.0
      600036      0       0.0
600036_clean      0       0.0
      600050      0       0.0
600050_clean      0       0.0
      600519      0       0.0
600519_clean      0       0.0
      601166      0       0.0
601166_clean      0       0.0
      601633      

# 3.数据合并

## 3.1 合并股票数据为长表

In [13]:
# ========== 保存清洗后的数据 ==========
print("\n" + "-"*40)
print("7. 保存清洗后的数据")
print("-"*40)

clean_folder = "data/clean"
if not os.path.exists(clean_folder):
    os.makedirs(clean_folder)

# 合并所有股票数据为长表
all_stocks_long = []

for code, df in cleaned_stocks.items():
    temp_df = df.reset_index()[['日期', '收盘', '成交量', '成交额', 'return', 'is_extreme']].copy()
    temp_df['code'] = code
    all_stocks_long.append(temp_df)

# 合并
combined_clean = pd.concat(all_stocks_long, ignore_index=True)

# 保存为 CSV
csv_path = os.path.join(clean_folder, "stock_clean.csv")
combined_clean.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f"✓ CSV 已保存: {csv_path}")
print(f"  形状: {combined_clean.shape}")

print("\n清洗后的数据预览：")
print(combined_clean.head(10))


----------------------------------------
7. 保存清洗后的数据
----------------------------------------
✓ CSV 已保存: data/clean\stock_clean.csv
  形状: (30316, 7)

清洗后的数据预览：
          日期           收盘        成交量           成交额    return  is_extreme  \
0 2020-01-02  3590.897899  101213040  3.342374e+09       NaN       False   
1 2020-01-03  3534.652262   80553629  2.584310e+09 -1.566339       False   
2 2020-01-06  3475.098059   87684058  2.761449e+09 -1.684867       False   
3 2020-01-07  3502.669449   57793343  1.827511e+09  0.793399       False   
4 2020-01-08  3493.846604   52999684  1.667144e+09 -0.251889       False   
5 2020-01-09  3551.195096   78373446  2.519928e+09  1.641414       False   
6 2020-01-10  3469.583781   55744373  1.762516e+09 -2.298137       False   
7 2020-01-13  3493.846604   62074912  1.957170e+09  0.699301       False   
8 2020-01-14  3461.863791   41971228  1.322401e+09 -0.915404       False   
9 2020-01-15  3362.606785   67294431  2.071444e+09 -2.867155       False   

  

## 3.2 宽表与长表转换

In [14]:
# ========== 3.2 宽表与长表转换 ==========
print("="*80)
print("3.2 宽表与长表转换")
print("="*80)

# ---------- 1. 创建宽表（收盘价）----------
print("\n【步骤1】将收盘价转换为宽表")

# 收集所有股票的收盘价
close_data = {}
for code, df in cleaned_stocks.items():
    close_data[code] = df['收盘']

# 创建宽表（日期为索引，每列一只股票）
wide_table = pd.DataFrame(close_data)
print(f"宽表形状: {wide_table.shape}")
print(f"日期范围: {wide_table.index.min()} 至 {wide_table.index.max()}")
print("\n宽表前5行：")
print(wide_table.head())

# ---------- 2. 转换回长表 ----------
print("\n【步骤2】使用 pd.melt 转换回长表")

# 重置索引，将日期变为列
wide_reset = wide_table.reset_index()
long_table = pd.melt(
    wide_reset,
    id_vars=['日期'],
    var_name='code',
    value_name='close'
)

# 删除缺失值
long_table = long_table.dropna()

print(f"长表形状: {long_table.shape}")
print("\n长表前10行：")
print(long_table.head(10))

# ---------- 验证转换正确性 ----------
print("\n【验证】转换前后数据一致性")
original_close = wide_table.stack().sort_index()
converted_close = long_table.set_index(['日期', 'code'])['close'].sort_index()
is_equal = (original_close == converted_close).all()
print(f"  转换前后数据一致: {is_equal}")

# ---------- 文字回答 ----------
print("\n" + "="*60)
print("【文字回答】宽表 vs 长表的适用场景")
print("="*60)
print("""
┌─────────────────────────────────────────────────────────────────┐
│  宽表（Wide Format）适用场景：                                   │
├─────────────────────────────────────────────────────────────────┤
│  1. 时间序列分析：计算多只股票的相关性矩阵                         │
│  2. 因子模型：CAPM、Fama-French 需要每只股票单独一列              │
│  3. 可视化：绘制多条时间序列折线图（如收盘价对比）                 │
│  4. 截面分析：比较同一时间点不同股票的指标                        │
│  5. 矩阵运算：协方差矩阵、投资组合优化                            │
├─────────────────────────────────────────────────────────────────┤
│  长表（Long Format）适用场景：                                    │
├─────────────────────────────────────────────────────────────────┤
│  1. 数据库存储：符合关系数据库的规范化设计（3NF）                  │
│  2. 分组操作：按 code 分组进行 groupby 聚合                       │
│  3. 多指标合并：与财务、宏观数据合并时更灵活                       │
│  4. 可视化：seaborn 的 hue 参数需要长表格式                       │
│  5. 机器学习：特征工程时每个样本是一行（日期+股票）                 │
└─────────────────────────────────────────────────────────────────┘
""")

3.2 宽表与长表转换

【步骤1】将收盘价转换为宽表
宽表形状: (1516, 20)
日期范围: 2020-01-02 00:00:00 至 2026-04-08 00:00:00

宽表前5行：
                 000002  000002_clean     000625  000625_clean       000858  \
日期                                                                            
2020-01-02  3590.897899   3590.897899  63.842483     63.842483  1955.521403   
2020-01-03  3534.652262   3534.652262  62.009967     62.009967  1932.868861   
2020-01-06  3475.098059   3475.098059  61.655287     61.655287  1912.881324   
2020-01-07  3502.669449   3502.669449  63.251349     63.251349  1915.398273   
2020-01-08  3493.846604   3493.846604  64.847411     64.847411  1908.291593   

            000858_clean      002352  002352_clean      600036  600036_clean  \
日期                                                                             
2020-01-02   1955.521403  129.552806    129.552806  164.510650    164.510650   
2020-01-03   1932.868861  128.550396    128.550396  166.710895    166.710895   
2020-01-06   1912.881324 

## 3.3 多表合并 - 加载指数数据

In [15]:
# ========== 3.3 多表合并 - 加载指数数据 ==========
print("="*80)
print("3.3 多表合并")
print("="*80)

# 加载指数数据
index_folder = "data/index"
index_files = glob.glob(os.path.join(index_folder, "index_*.csv"))

index_data = {}

for file_path in index_files:
    filename = os.path.basename(file_path)
    # 提取代码: index_000300.csv -> 000300
    code = filename.replace("index_", "").replace(".csv", "")
    
    df = pd.read_csv(file_path, encoding='utf-8-sig')
    df['日期'] = pd.to_datetime(df['日期'])
    df.set_index('日期', inplace=True)
    
    index_data[code] = df['收盘']
    print(f"加载指数 {code}: {len(df)} 条记录")

# 创建指数宽表
index_wide = pd.DataFrame(index_data)
index_wide.columns = [f"index_{col}" for col in index_wide.columns]
print(f"\n指数宽表形状: {index_wide.shape}")

3.3 多表合并
加载指数 000300: 1516 条记录
加载指数 000905: 1516 条记录

指数宽表形状: (1516, 2)


In [16]:
# ========== 个股与指数合并（left join）==========
print("\n" + "-"*40)
print("个股与指数合并")
print("-"*40)

# 获取个股长表
stock_long = combined_clean.copy()
stock_long['日期'] = pd.to_datetime(stock_long['日期'])

# 重置指数索引，准备合并
index_long = index_wide.reset_index()

# Left join: 保留所有个股数据
merged_stock_index = stock_long.merge(
    index_long,
    on='日期',
    how='left'
)

print(f"合并前个股数据行数: {len(stock_long)}")
print(f"合并后数据行数: {len(merged_stock_index)}")
print(f"指数数据缺失行数: {merged_stock_index['index_000300'].isna().sum()}")

print("\n【行数变化说明】")
print("使用 left join 保留所有个股数据，指数数据缺失是因为：")
print("1. 指数交易日与个股一致，通常无缺失")
print("2. 若有个股上市日期晚于指数起始日期，前期指数为 NaN")
print("3. 本项目中指数和个股时间范围一致，无缺失")


----------------------------------------
个股与指数合并
----------------------------------------
合并前个股数据行数: 30316
合并后数据行数: 30316
指数数据缺失行数: 0

【行数变化说明】
使用 left join 保留所有个股数据，指数数据缺失是因为：
1. 指数交易日与个股一致，通常无缺失
2. 若有个股上市日期晚于指数起始日期，前期指数为 NaN
3. 本项目中指数和个股时间范围一致，无缺失


In [17]:
# ========== 加载宏观数据 ==========
print("\n" + "-"*40)
print("加载宏观数据")
print("-"*40)

macro_folder = "data/macro"
macro_data = {}

if os.path.exists(macro_folder):
    macro_files = glob.glob(os.path.join(macro_folder, "macro_*.csv"))
    
    for file_path in macro_files:
        filename = os.path.basename(file_path)
        name = filename.replace("macro_", "").replace(".csv", "")
        
        df = pd.read_csv(file_path, encoding='utf-8-sig')
        
        # 统一日期格式
        if '日期' in df.columns:
            df['日期'] = pd.to_datetime(df['日期'])
            # 转换为月度标识（YYYY-MM）
            df['year_month'] = df['日期'].dt.strftime('%Y-%m')
            macro_data[name] = df
            
            print(f"加载宏观指标 {name}: {len(df)} 条记录")
            print(f"  列: {df.columns.tolist()}")
else:
    print("宏观数据文件夹不存在，跳过")


----------------------------------------
加载宏观数据
----------------------------------------
加载宏观指标 cpi: 70 条记录
  列: ['日期', 'CPI同比(%)', 'year_month']
加载宏观指标 lpr: 75 条记录
  列: ['日期', 'LPR_1Y(%)', 'LPR_5Y(%)', 'year_month']
加载宏观指标 m2: 74 条记录
  列: ['日期', 'M2同比(%)', 'year_month']


In [18]:
# ========== 宏观数据与日度数据合并 ==========
print("\n" + "-"*40)
print("宏观数据与日度数据合并")
print("-"*40)

# 为日度数据添加年月列
merged_data = merged_stock_index.copy()
merged_data['year_month'] = merged_data['日期'].dt.strftime('%Y-%m')

print(f"日度数据行数: {len(merged_data)}")
print(f"日期范围: {merged_data['日期'].min()} 至 {merged_data['日期'].max()}")
print(f"月度覆盖: {merged_data['year_month'].nunique()} 个月")

# 合并 CPI 数据
if 'cpi' in macro_data:
    cpi_df = macro_data['cpi'][['year_month', 'CPI同比(%)']].copy()
    cpi_df = cpi_df.drop_duplicates(subset=['year_month'])
    
    merged_data = merged_data.merge(cpi_df, on='year_month', how='left')
    print(f"\n合并CPI后行数: {len(merged_data)}")
    print(f"CPI缺失行数: {merged_data['CPI同比(%)'].isna().sum()}")

# 合并汇率数据
if 'usd_cny' in macro_data:
    usd_df = macro_data['usd_cny'][['year_month', '美元兑人民币']].copy()
    usd_df = usd_df.drop_duplicates(subset=['year_month'])
    
    merged_data = merged_data.merge(usd_df, on='year_month', how='left')
    print(f"合并汇率后行数: {len(merged_data)}")
    print(f"汇率缺失行数: {merged_data['美元兑人民币'].isna().sum()}")

print("\n【频率不一致处理说明】")
print("宏观数据为月度频率，股票数据为日度频率。")
print("处理方式：将月度数据映射到对应月份的每个交易日")
print("具体实现：为日度数据添加 year_month 列，与宏观数据的年月进行匹配")
print("优点：保留了日度数据的粒度，同时引入了月度宏观信息")


----------------------------------------
宏观数据与日度数据合并
----------------------------------------
日度数据行数: 30316
日期范围: 2020-01-02 00:00:00 至 2026-04-08 00:00:00
月度覆盖: 76 个月

合并CPI后行数: 30316
CPI缺失行数: 2860

【频率不一致处理说明】
宏观数据为月度频率，股票数据为日度频率。
处理方式：将月度数据映射到对应月份的每个交易日
具体实现：为日度数据添加 year_month 列，与宏观数据的年月进行匹配
优点：保留了日度数据的粒度，同时引入了月度宏观信息


In [ ]:
# ========== 保存合并后的综合数据 ==========
print("\n" + "-"*40)
print("保存合并后的综合数据")
print("-"*40)

combined_folder = "data/combined"
if not os.path.exists(combined_folder):
    os.makedirs(combined_folder)

# 删除临时列
if 'year_month' in merged_data.columns:
    merged_data = merged_data.drop(columns=['year_month'])

# 保存
combined_path = os.path.join(combined_folder, "combined_data.csv")
merged_data.to_csv(combined_path, index=False, encoding='utf-8-sig')

print(f"✓ 综合数据已保存: {combined_path}")
print(f"  最终数据形状: {merged_data.shape}")
print(f"  列名: {merged_data.columns.tolist()}")
print("\n数据预览：")
print(merged_data.head(10))

## 3.4 额外保存数据：Parquet 格式

In [1]:
# ========== Parquet 格式（简化版）==========
print("="*80)
print("3.4 方式 B：Parquet 格式（简化版）")
print("="*80)

import pandas as pd
import os
import time

# 路径定义
clean_folder = "data/clean"
clean_csv_path = os.path.join(clean_folder, "stock_clean.csv")
clean_parquet_path = os.path.join(clean_folder, "stock_clean.parquet")

# 读取数据
df = pd.read_csv(clean_csv_path, encoding='utf-8-sig')
df['日期'] = pd.to_datetime(df['日期'])
print(f"数据形状: {df.shape}")
print(f"列名: {df.columns.tolist()}")

# 保存为 Parquet
df.to_parquet(clean_parquet_path, index=False)
print(f"\n✓ Parquet 已保存: {clean_parquet_path}")

# 验证可以读取
df_check = pd.read_parquet(clean_parquet_path)
print(f"✓ 验证读取成功，形状: {df_check.shape}")

# 性能对比
t0 = time.time()
pd.read_csv(clean_csv_path, encoding='utf-8-sig')
csv_time = time.time() - t0

t0 = time.time()
pd.read_parquet(clean_parquet_path)
parquet_time = time.time() - t0

csv_size = os.path.getsize(clean_csv_path) / 1024
parquet_size = os.path.getsize(clean_parquet_path) / 1024

print(f"\n性能对比:")
print(f"  CSV: {csv_time:.3f}s, {csv_size:.1f} KB")
print(f"  Parquet: {parquet_time:.3f}s, {parquet_size:.1f} KB")
print(f"  速度提升: {csv_time/parquet_time:.1f} 倍")
print(f"  体积减少: {(1 - parquet_size/csv_size)*100:.1f}%")

print("\n✅ Parquet 格式处理完成")

3.4 方式 B：Parquet 格式（简化版）
数据形状: (30316, 7)
列名: ['日期', '收盘', '成交量', '成交额', 'return', 'is_extreme', 'code']

✓ Parquet 已保存: data/clean\stock_clean.parquet
✓ 验证读取成功，形状: (30316, 7)

性能对比:
  CSV: 0.101s, 2428.6 KB
  Parquet: 0.017s, 619.2 KB
  速度提升: 6.0 倍
  体积减少: 74.5%

✅ Parquet 格式处理完成


In [21]:
# ========== 验证所有生成的文件 ==========
print("="*80)
print("验证所有生成的文件")
print("="*80)

files_to_check = [
    "data/clean/stock_clean.csv",
    "data/clean/stock_clean.parquet",
    "data/combined/combined_data.csv"
]

for file_path in files_to_check:
    if os.path.exists(file_path):
        size = os.path.getsize(file_path) / 1024
        print(f"✓ {file_path} ({size:.1f} KB)")
    else:
        print(f"✗ {file_path} 不存在")

print("\n" + "="*80)
print("✅ 02_clean.ipynb 执行完成！")
print("="*80)

验证所有生成的文件
✓ data/clean/stock_clean.csv (2428.6 KB)
✓ data/clean/stock_clean.parquet (619.2 KB)
✓ data/combined/combined_data.csv (3134.9 KB)

✅ 02_clean.ipynb 执行完成！


In [3]:
# ========== 将 Parquet 文件的列名改为英文 ==========
import pandas as pd

# 读取 Parquet 文件
df = pd.read_parquet("data/clean/stock_clean.parquet")

# 查看当前列名
print("当前列名:", df.columns.tolist())

# 重命名列（根据实际列名调整）
column_mapping = {
    '日期': 'date',
    'code': 'code',
    '收盘': 'close',
    '开盘': 'open',
    '最高': 'high',
    '最低': 'low',
    '成交量': 'volume',
    '成交额': 'amount',
    'return': 'return',
    'is_extreme': 'is_extreme'
}

# 只重命名存在的列
rename_dict = {k: v for k, v in column_mapping.items() if k in df.columns}
df_renamed = df.rename(columns=rename_dict)

print("\n重命名后列名:", df_renamed.columns.tolist())

# 保存为新的 Parquet 文件（覆盖原文件）
df_renamed.to_parquet("data/clean/stock_clean.parquet", index=False)
print("\n✓ 已重命名列并保存")

# 现在可以用英文列名读取了
df_test = pd.read_parquet("data/clean/stock_clean.parquet",
                          columns=["date", "code", "close"])
print("\n测试读取成功:")
print(df_test.head())

当前列名: ['date', 'close', 'volume', 'amount', 'return', 'is_extreme', 'code']

重命名后列名: ['date', 'close', 'volume', 'amount', 'return', 'is_extreme', 'code']

✓ 已重命名列并保存

测试读取成功:
        date    code        close
0 2020-01-02  000002  3590.897899
1 2020-01-03  000002  3534.652262
2 2020-01-06  000002  3475.098059
3 2020-01-07  000002  3502.669449
4 2020-01-08  000002  3493.846604


## 3.5 csv与parquet性能对比

In [4]:
import pandas as pd, pyarrow.parquet as pq, os, time

# 列式读取（只加载需要的列）
df = pd.read_parquet("data/clean/stock_clean.parquet",
                     columns=["date", "code", "close"])

# 查看 Schema（类型契约）
schema = pq.read_schema("data/clean/stock_clean.parquet")
print(schema)

# 与 CSV 对比：读取速度与文件体积
t0 = time.time()
pd.read_csv("data/clean/stock_clean.csv")
print(f"CSV  读取耗时: {time.time()-t0:.3f}s  "
      f"文件大小: {os.path.getsize('data/clean/stock_clean.csv')/1024:.1f} KB")

t0 = time.time()
pd.read_parquet("data/clean/stock_clean.parquet")
print(f"Parquet 读取耗时: {time.time()-t0:.3f}s  "
      f"文件大小: {os.path.getsize('data/clean/stock_clean.parquet')/1024:.1f} KB")

date: timestamp[ns]
close: double
volume: int64
amount: double
return: double
is_extreme: bool
code: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 861
CSV  读取耗时: 0.068s  文件大小: 2428.6 KB
Parquet 读取耗时: 0.014s  文件大小: 619.0 KB
